# NB4 — 4D-Humans (HMR 2.0) per-frame 자세 init **[GPU]**
기존 `4d_humans_colab_WALKING_v1` 노트북과 같은 로직 (detectron2 ViTDet 검출 → primary subject → HMR2).
바뀐 것: SMPL 업로드 대신 `assets/` 사용, 입력 = NB1 `00_extract/rgb/*.jpg`, 출력 = 드라이브 `03_hmr2/hmr2_walking_track.npz`.

A0(NB5) 가 읽는 것: `fids`, `global_orient_aa[T,3]`, `body_pose_aa[T,23,3]`. betas 는 진단용 (체형은 STAND β 고정).

In [ ]:
#@title 0. 설정 — 세션마다 여기만 바꾸면 됨
ROOT    = "/content/drive/MyDrive/walking_0729"   #@param {type:"string"}
SESSION = "0729_walk_3kmh"                        #@param {type:"string"}

from google.colab import drive
drive.mount("/content/drive")
import os, sys, importlib
os.environ["WALK_ROOT"] = ROOT
os.environ["WALK_SESSION"] = SESSION
sys.path.insert(0, f"{ROOT}/scripts")
import walkcfg; importlib.reload(walkcfg)
assert os.path.isdir(f"{ROOT}/scripts"), f"{ROOT}/scripts 없음 — colab_kit 업로드 위치 확인"
print("세션 폴더:", walkcfg.SESS)

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "런타임을 GPU로 바꿔주세요 (런타임 → 런타임 유형 변경 → T4 GPU)"

## 2. 설치 (~5–10분)

In [ ]:
# 4D-Humans 클론 + 설치
!git clone --recursive https://github.com/shubham-goel/4D-Humans.git
%cd 4D-Humans
!pip install -e .[all] -q

# detectron2 (사람 검출용). .[all]에 포함 안 되면 아래가 설치함.
!pip install "git+https://github.com/facebookresearch/detectron2.git" -q

# 대체안 (위가 실패할 때만 주석 해제):
# !python -m pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu121/torch2.1/index.html

In [ ]:
#@title 3. SMPL neutral 배치 (assets 에서 복사, chumpy 제거본)
import shutil
from pathlib import Path
os.makedirs("data", exist_ok=True)
shutil.copy(f"{ROOT}/assets/basicModel_neutral_lbs_10_207_0_v1.0.0.pkl", "data/basicModel_neutral_lbs_10_207_0_v1.0.0.pkl")
cache_smpl = Path.home()/".cache"/"4DHumans"/"data"/"smpl"; cache_smpl.mkdir(parents=True, exist_ok=True)
shutil.copy(f"{ROOT}/assets/models/smpl/SMPL_NEUTRAL.pkl", cache_smpl/"SMPL_NEUTRAL.pkl")
print("SMPL 배치 완료")

In [ ]:
#@title 4. 입력 프레임 = NB1 00_extract/rgb
import re
from pathlib import Path
RGB = Path(walkcfg.EXTRACT_DIR) / "rgb"
FRAMES = sorted([(int(p.stem), p) for p in RGB.glob("*.jpg")], key=lambda x: x[0])
assert FRAMES, f"{RGB} 비어있음 → NB1 먼저"
fids = [f for f, _ in FRAMES]
gaps = [b-a for a, b in zip(fids[:-1], fids[1:]) if b-a != 1]
print(f"프레임 {len(FRAMES)} | fid {fids[0]}~{fids[-1]} | 불연속 {len(gaps)}")

## Cell 5 — demo.py overlay 서브셋 검수 (권장)

250장 전부는 비현실적 → **보행 주기 걸치게 매 N프레임만** overlay로 spot-check.

확인할 것: HMR2가 걷는 자세를 대략 맞게 잡는지 / 좌우·앞뒤 뒤집힘 없는지 / 크게 잘린 프레임 없는지.
여기서 살짝 어긋나도 A0가 depth로 보정합니다.

In [ ]:
import os, glob, shutil
import matplotlib.pyplot as plt, matplotlib.image as mpimg

STRIDE = 25  # 매 25프레임 (~10장)
sub = Path("overlay_subset"); shutil.rmtree(sub, ignore_errors=True); sub.mkdir()
for fid, p in FRAMES[::STRIDE]:
    shutil.copy(str(p), str(sub / p.name))

shutil.rmtree("demo_out", ignore_errors=True)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python demo.py \
    --img_folder overlay_subset --out_folder demo_out \
    --batch_size 8 --side_view --full_frame

outs = sorted(glob.glob("demo_out/*.png"))
if not outs:
    raise RuntimeError("demo_out 비어있음. 위 로그 첫 Traceback이 원인.")
plt.figure(figsize=(14, 4*((len(outs)+1)//2)))
for i, pth in enumerate(outs):
    plt.subplot((len(outs)+1)//2, 2, i+1); plt.imshow(mpimg.imread(pth))
    plt.axis("off"); plt.title(os.path.basename(pth), fontsize=8)
plt.tight_layout(); plt.show()

## Cell 6 — per-frame HMR2 → walking 자세 궤적 저장

프레임마다 primary subject 1명 골라 HMR2 회귀. **STAND의 per-frame 방식 그대로**,
루프 대상만 4장→250프레임, 저장을 per-view 묶음→**fid 인덱싱 궤적**으로 변경.

⚠ betas는 **진단용만** 저장(프레임마다 출렁이므로 사용 금지). 체형은 STAND freeze 고정.

In [ ]:
import os, json, shutil
from pathlib import Path
import cv2, numpy as np, torch
from scipy.spatial.transform import Rotation as Rot

# PyTorch 2.6+ 신뢰된 공식 checkpoint 로드 호환 패치
if not getattr(torch.load, "_hmr2_weights_only_patched", False):
    _orig = torch.load
    def _patched(*a, **k):
        k.setdefault("weights_only", False); return _orig(*a, **k)
    _patched._hmr2_weights_only_patched = True
    torch.load = _patched

from hmr2.models import load_hmr2, download_models, DEFAULT_CHECKPOINT
from hmr2.utils import recursive_to
from hmr2.datasets.vitdet_dataset import ViTDetDataset
from hmr2.utils.utils_detectron2 import DefaultPredictor_Lazy
import hmr2
from detectron2.config import LazyConfig

device = torch.device("cuda")

download_models()
model, model_cfg = load_hmr2(DEFAULT_CHECKPOINT)
model = model.to(device).eval()

cfg_path = Path(hmr2.__file__).parent / "configs" / "cascade_mask_rcnn_vitdet_h_75ep.py"
dcfg = LazyConfig.load(str(cfg_path))
dcfg.train.init_checkpoint = (
    "https://dl.fbaipublicfiles.com/detectron2/ViTDet/COCO/"
    "cascade_mask_rcnn_vitdet_h/f328730692/model_final_f05665.pkl"
)
for i in range(3):
    dcfg.model.roi_heads.box_predictors[i].test_score_thresh = 0.25
detector = DefaultPredictor_Lazy(dcfg)

def rotmat_to_aa(rm):
    s = rm.shape[:-2]
    return Rot.from_matrix(rm.reshape(-1,3,3)).as_rotvec().reshape(*s,3).astype(np.float32)

def choose_primary(boxes, scores, hw):
    h, w = hw
    wh = np.maximum(boxes[:,2:4]-boxes[:,0:2], 1.0)
    area = wh[:,0]*wh[:,1]
    ctr = 0.5*(boxes[:,0:2]+boxes[:,2:4])
    cd = np.linalg.norm(ctr - np.array([w*0.5,h*0.5])[None], axis=1)/max(np.hypot(w,h),1.0)
    rank = scores*np.sqrt(area)*(1.0-0.35*np.clip(cd,0,1))
    return int(np.argmax(rank)), rank

OUT = Path(walkcfg.FOURDH_SEQ).parent; OUT.mkdir(parents=True, exist_ok=True)   # [colab_kit] 드라이브 03_hmr2/

T = len(FRAMES)
fids_arr        = np.zeros(T, np.int32)
go_arr          = np.zeros((T,3), np.float32)
bp_arr          = np.zeros((T,23,3), np.float32)
betas_arr       = np.zeros((T,10), np.float32)   # 진단용 (사용 금지)
cam_arr         = np.zeros((T,3), np.float32)
bbox_arr        = np.zeros((T,4), np.float32)
score_arr       = np.zeros(T, np.float32)
count_arr       = np.zeros(T, np.int32)
hw_arr          = np.zeros((T,2), np.int32)
prev_ctr = None
switch_warn = 0

for i,(fid,path) in enumerate(FRAMES):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None: raise RuntimeError(f"읽기 실패: {path}")
    h,w = img.shape[:2]
    inst = detector(img)["instances"].to("cpu")
    valid = (inst.pred_classes.numpy()==0) & (inst.scores.numpy()>=0.50)
    boxes = inst.pred_boxes.tensor.numpy()[valid]; scores = inst.scores.numpy()[valid]
    if len(boxes)==0: raise RuntimeError(f"[검출 실패] {path.name}")
    pidx, _ = choose_primary(boxes, scores, (h,w))
    box = boxes[pidx:pidx+1]

    # identity switch 감시 (트레드밀 단일 피사체라 bbox 중심이 급변하면 경고)
    ctr = 0.5*(box[0,0:2]+box[0,2:4])
    if prev_ctr is not None and np.linalg.norm(ctr-prev_ctr) > 0.5*np.hypot(w,h):
        switch_warn += 1
    prev_ctr = ctr

    ds = ViTDetDataset(model_cfg, img, box)
    ld = torch.utils.data.DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    with torch.inference_mode():
        b = recursive_to(next(iter(ld)), device); out = model(b)
    smpl = out["pred_smpl_params"]
    go = rotmat_to_aa(smpl["global_orient"].cpu().numpy()[0]).reshape(3)
    bp = rotmat_to_aa(smpl["body_pose"].cpu().numpy()[0]).reshape(23,3)

    fids_arr[i]=fid; go_arr[i]=go; bp_arr[i]=bp
    betas_arr[i]=smpl["betas"].cpu().numpy()[0]
    cam_arr[i]=out["pred_cam"].cpu().numpy()[0]
    bbox_arr[i]=box[0]; score_arr[i]=scores[pidx]; count_arr[i]=len(boxes); hw_arr[i]=[h,w]

    if i % 25 == 0 or i == T-1:
        print(f"[{i+1}/{T}] fid={fid} score={scores[pidx]:.2f} persons={len(boxes)}")

# ---- 궤적 번들 저장 (A0가 읽는 포맷) ----
np.savez_compressed(
    OUT / "hmr2_walking_track.npz",
    schema_version=np.array("hmr2_walking_track_v1"),
    fids=fids_arr,
    global_orient_aa=go_arr,        # [T,3]  A0 init
    body_pose_aa=bp_arr,            # [T,23,3] A0 init
    betas_per_frame=betas_arr,      # [T,10] 진단용 — 사용 금지(STAND freeze 사용)
    pred_cam_weakpersp=cam_arr,     # [T,3]  metric 아님
    bbox_xyxy=bbox_arr, detector_score=score_arr,
    detected_person_count=count_arr, source_rgb_hw=hw_arr,
)

# ---- β 분산 = "왜 walking betas를 버리고 STAND를 freeze하는가"의 측정 증거 ----
b_std = betas_arr.std(0); b_mean = betas_arr.mean(0)
manifest = {
    "schema_version": "hmr2_walking_track_v1",
    "purpose": "Per-frame HMR2 init for A0 alignment. betas are DIAGNOSTIC ONLY.",
    "n_frames": int(T),
    "fid_range": [int(fids_arr.min()), int(fids_arr.max())],
    "betas_note": "DO NOT USE. Shape comes from STAND final_single_smpl_params.npz (frozen).",
    "betas_frame_std": [float(x) for x in b_std],
    "betas_frame_mean": [float(x) for x in b_mean],
    "identity_switch_warnings": int(switch_warn),
    "next_step": "A0 alignment: read fids/global_orient_aa/body_pose_aa as init; freeze STAND betas.",
}
with open(OUT / "hmr2_walking_manifest.json","w",encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("\n완료:", OUT / "hmr2_walking_track.npz")
print("betas per-frame std (클수록 프레임 간 출렁 = freeze 정당화):", np.round(b_std,3))
print("identity switch 경고 프레임:", switch_warn)

## 7. 결과 확인

In [ ]:
import numpy as np, json
from pathlib import Path

OUT = Path(walkcfg.FOURDH_SEQ).parent
bd = np.load(OUT / "hmr2_walking_track.npz", allow_pickle=False)
print("schema:", bd["schema_version"])
print("fids:", bd["fids"].shape, f"({int(bd['fids'].min())}~{int(bd['fids'].max())})")
print("global_orient_aa:", bd["global_orient_aa"].shape)
print("body_pose_aa:", bd["body_pose_aa"].shape)
print("betas_per_frame(진단용):", bd["betas_per_frame"].shape)

# A0 sanity: 방향 init이 프레임 간 심하게 튀지 않는지 (걷기 한 방향이면 완만해야)
go = bd["global_orient_aa"]
dgo = np.linalg.norm(np.diff(go, axis=0), axis=1)
print(f"\nglobal_orient 프레임간 변화(rad): median={np.median(dgo):.3f} max={dgo.max():.3f}")
print("  (max가 과도하면 A2 temporal 부담↑ 또는 detection 튐 — 해당 fid 점검)")

with open(OUT / "hmr2_walking_manifest.json", encoding="utf-8") as f:
    man = json.load(f)
print("\nβ frame std:", np.round(man["betas_frame_std"],3))
print("identity switch 경고:", man["identity_switch_warnings"])

print("saved:", walkcfg.FOURDH_SEQ)

## 시각화 (NB4)

In [ ]:
#@title 시각화 — NB4 결과 확인 (viz/NB4/  + viz/videos/)
FIDS = "5660 5750"   #@param {type:"string"}
N_EVEN = 0           #@param {type:"integer"}
# 셀 0 을 안 돌렸어도 혼자 동작하도록 (드라이브 마운트 + 세션 설정)
import os, sys, importlib
if "walkcfg" not in globals():
    ROOT = "/content/drive/MyDrive/walking_0729"   # 셀 0 과 같게
    SESSION = "0729_full_3kmh"                     # 셀 0 과 같게
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["WALK_ROOT"] = ROOT; os.environ["WALK_SESSION"] = SESSION
    sys.path.insert(0, f"{ROOT}/scripts"); import walkcfg
print("세션:", walkcfg.SESS)
MAKE_VIDEO = True    #@param {type:"boolean"}
FPS = 15             #@param {type:"number"}
!pip -q install smplx imageio-ffmpeg
!cd "{ROOT}/scripts" && python viz_stages.py --nb 4 --fids {FIDS} --n {N_EVEN}
if MAKE_VIDEO:
    !cd "{ROOT}/scripts" && python viz_video.py --nb 4 --fps {FPS}
from IPython.display import Image, display
import glob, os
for p in sorted(glob.glob(f"{walkcfg.SESS}/viz/NB4/*.png")):
    print(os.path.basename(p)); display(Image(p, width=1100))
from IPython.display import Video
_mp4 = f"{walkcfg.SESS}/viz/videos/NB4_4dh.mp4"
if os.path.exists(_mp4):
    print("영상:", _mp4, f"{os.path.getsize(_mp4)/1e6:.1f} MB  (노트북에 넣어 재생 — 확인 후 출력 지우기 권장)")
    display(Video(_mp4, embed=True, width=1100))